### Phase Two: Predictive Risk Modeling

Step 1: Set up your Python Environmen

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# 1. Load the cleaned data
df = pd.read_csv('Kenya_Insurance_Analysis_Cleaned.csv')

# 2. Create the Target: 1 if there was a claim, 0 if no claim
# This is what the model will try to predict for new customers
df['Claim_Occurred'] = (df['Claims_Frequency'] > 0).astype(int)

# 3. Check the balance of your data
print("Claim Counts (0 = No Claim, 1 = Claim):")
print(df['Claim_Occurred'].value_counts())
print("\nFirst 5 rows of the new target:")
print(df[['Policy_ID', 'Claims_Frequency', 'Claim_Occurred']].head())

Claim Counts (0 = No Claim, 1 = Claim):
Claim_Occurred
0    1338
1     662
Name: count, dtype: int64

First 5 rows of the new target:
    Policy_ID  Claims_Frequency  Claim_Occurred
0  P202300001                 1               1
1  P202300002                 0               0
2  P202300003                 0               0
3  P202300004                 0               0
4  P202300005                 0               0


Step 2: Feature Encoding and Model Training

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# 1. Select your Features (Predictors)
features = ['Customer_Age', 'Vehicle_Age', 'Vehicle_Engine_Capacity', 
            'Driver_Experience_Years', 'Gender', 'Region', 'Vehicle_Type', 'Use_Purpose']

X = df[features].copy()
y = df['Claim_Occurred']

# 2. Encode Categorical Columns
# This turns "Male/Female" into 0/1, and Regions into 0,1,2,3...
le = LabelEncoder()
categorical_cols = ['Gender', 'Region', 'Vehicle_Type', 'Use_Purpose']

for col in categorical_cols:
    X[col] = le.fit_transform(X[col])

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and Train the Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 5. Look at the Coefficients (The Importance of each feature)
importance = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("Model Feature Coefficients:")
print(importance)

Model Feature Coefficients:
                   Feature  Coefficient
4                   Gender     0.068864
6             Vehicle_Type     0.051275
0             Customer_Age     0.007355
1              Vehicle_Age     0.002144
2  Vehicle_Engine_Capacity     0.000076
5                   Region    -0.006975
3  Driver_Experience_Years    -0.007559
7              Use_Purpose    -0.032234


c:\Users\leovo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Which factors drive risk in your model?

- Positive Coefficient: As this value goes up, the probability of a claim increases.

- Negative Coefficient: As this value goes up, the probability of a claim decreases (e.g., more experience usually equals lower risk).

Step 3: Scaling and Model Evaluation

We need to "standardize" the data (give everything a mean of 0 and a variance of 1) so the model converges properly. We will also check the Accuracy Score to see how often the model is right

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Scaling the numerical features (Essential for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Re-train the model on scaled data
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 3. Make predictions on the Test Set
y_pred = model.predict(X_test_scaled)

# 4. Check the performance
print("--- Model Performance ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

# 5. Get the clean coefficients again
importance_scaled = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("\nImproved Feature Coefficients (Scaled):")
print(importance_scaled)

--- Model Performance ---
Accuracy Score: 64.00%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.64      1.00      0.78       256
           1       0.00      0.00      0.00       144

    accuracy                           0.64       400
   macro avg       0.32      0.50      0.39       400
weighted avg       0.41      0.64      0.50       400


Improved Feature Coefficients (Scaled):
                   Feature  Coefficient
0             Customer_Age     0.096607
6             Vehicle_Type     0.062300
2  Vehicle_Engine_Capacity     0.052830
4                   Gender     0.037859
1              Vehicle_Age     0.012518
5                   Region    -0.015784
7              Use_Purpose    -0.022047
3  Driver_Experience_Years    -0.099824


c:\Users\leovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\leovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\leovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### The Statistical Problem:
- My model has an accuracy of 64%, but your classification report shows a Recall of 0.00 for Class 1.

- What happened: Because 64% of your data is "No Claim" (Class 0), the model discovered it can get 64% accuracy by simply guessing "No Claim" for every single person.

- The Result: It failed to identify even one actual claimant. In insurance, this model is useless because it doesn't find the high-risk people!

### The Solution: Class Balancing & Probability Scoring
To fix this, we will do two things:

1. class_weight='balanced': This tells the model to penalize errors on "Claims" more heavily so it stops ignoring them.

2. predict_proba: Instead of a simple "Yes/No," we will calculate the Probability Score (0 to 1). In a real insurance company, we don't just want to know if they will claim, but how likely they are to claim.

In [ ]:
# 1. Re-train the model with 'balanced' weights to handle the majority class bias
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# 2. Make predictions again
y_pred = model.predict(X_test_scaled)

# 3. Get the Probability Scores (This is the "Secret Sauce" for your Dashboard)
# predict_proba returns [prob_of_no_claim, prob_of_claim]. we take index 1.
y_probs = model.predict_proba(X_test_scaled)[:, 1]

print("--- Improved Model Performance (Balanced) ---")
print(classification_report(y_test, y_pred))

# 4. Now, let's apply this model to your ENTIRE dataset to get scores for Power BI
all_features_scaled = scaler.transform(X) # Scale the whole dataset
df['Claim_Probability'] = model.predict_proba(all_features_scaled)[:, 1]

# 5. Create a Risk Rating based on the probability
def categorize_risk(prob):
    if prob < 0.4: return 'Low Risk'
    elif prob < 0.7: return 'Medium Risk'
    else: return 'High Risk'

df['Predicted_Risk_Level'] = df['Claim_Probability'].apply(categorize_risk)

print("\nSample of the New Predictive Dataset:")
print(df[['Policy_ID', 'Claim_Probability', 'Predicted_Risk_Level']].head())

# 6. Save this as my new "Phase 2" file
df.to_csv('Kenya_Insurance_With_Predictions.csv', index=False)
print("\nSuccess! 'Kenya_Insurance_With_Predictions.csv' is ready for Power BI.")

--- Improved Model Performance (Balanced) ---
              precision    recall  f1-score   support

           0       0.61      0.42      0.50       256
           1       0.33      0.51      0.40       144

    accuracy                           0.46       400
   macro avg       0.47      0.47      0.45       400
weighted avg       0.51      0.46      0.46       400


Sample of the New Predictive Dataset:
    Policy_ID  Claim_Probability Predicted_Risk_Level
0  P202300001           0.517637          Medium Risk
1  P202300002           0.503115          Medium Risk
2  P202300003           0.516871          Medium Risk
3  P202300004           0.525156          Medium Risk
4  P202300005           0.458436          Medium Risk

Success! 'Kenya_Insurance_With_Predictions.csv' is ready for Power BI.


### The Statistical Interpretation
Look at your Recall for Class 1 (0.51). This means your model is now successfully identifying 51% of all potential claimants. While the accuracy dropped to 46%, in the insurance world, this is often a better trade-off. It is better to flag a risky driver who doesn't crash (a "False Positive") than to let a dangerous driver join the pool without a warning (a "False Negative").

In [5]:
# Check the last 5 rows of the new file
check_df = pd.read_csv('Kenya_Insurance_With_Predictions.csv')
print(check_df[['Policy_ID', 'Claim_Probability', 'Predicted_Risk_Level']].tail())

       Policy_ID  Claim_Probability Predicted_Risk_Level
1995  P202400996           0.487981          Medium Risk
1996  P202400997           0.538939          Medium Risk
1997  P202400998           0.513522          Medium Risk
1998  P202400999           0.485750          Medium Risk
1999  P202401000           0.502310          Medium Risk


In [6]:
print("Minimum Probability:", df['Claim_Probability'].min())
print("Maximum Probability:", df['Claim_Probability'].max())
print("Average Probability:", df['Claim_Probability'].mean())

Minimum Probability: 0.4307264556261919
Maximum Probability: 0.559888155611726
Average Probability: 0.49981960825634564


In [7]:
# We will use quantiles to ensure we get a mix of Low, Medium, and High
# This divides the data into 3 equal-sized groups based on risk
df['Predicted_Risk_Level'] = pd.qcut(df['Claim_Probability'], 
                                     q=3, 
                                     labels=['Low Risk', 'Medium Risk', 'High Risk'])

print("New Risk Level Counts:")
print(df['Predicted_Risk_Level'].value_counts())

# Save the updated file
df.to_csv('Kenya_Insurance_With_Predictions.csv', index=False)

New Risk Level Counts:
Predicted_Risk_Level
Low Risk       667
High Risk      667
Medium Risk    666
Name: count, dtype: int64
